# Evaluating and Refining an AI-Assisted SQL Workflow

**Scenario:** Before the team uses my appointment analysis to make decisions, my manager asked: *"Can you confirm that the queries are actually correct and reliable?"* This lab reviews the SQL workflow from the previous lab (*Foundations of Prompt Engineering for SQL Workflows*), validates it, finds where it could mislead the team, and improves it with a prompt chain.

**AI assistant used:** Claude (Anthropic). Every AI response and every piece of AI-generated SQL below is saved without manual edits.

**Tools:** Python `sqlite3` + pandas, with `healthcare_operations.db`.

## Setup

In [1]:
import re
import sqlite3
import pandas as pd

conn = sqlite3.connect("healthcare_operations.db")
pd.read_sql("SELECT * FROM reporting_context;", conn).T

,0
context_id,1
reporting_date,2025-06-29
data_start_date,2025-01-06
data_end_date,2025-06-29
recent_30_day_start,2025-05-30
recent_60_day_start,2025-04-30
recent_3_month_start,2025-03-29
recent_6_month_start,2024-12-29
primary_date_column,appointments.appointment_date
note,Use reporting_date instead of date('now') beca...


In [2]:
# Schema reference: the source of truth for every column check in this lab
SCHEMA = {
    t: set(pd.read_sql(f"PRAGMA table_info({t});", conn)["name"])
    for t in ["patients", "appointments", "providers", "clinics", "reporting_context"]
}


def invalid_column_refs(sql):
    """Return alias.column references that point at a real table but use a column not in that table."""
    aliases = {alias: table for table, alias in
               re.findall(r"(?:FROM|JOIN)\s+(\w+)(?:\s+AS)?\s+(\w+)", sql, flags=re.I)
               if table in SCHEMA}
    refs = re.findall(r"\b(\w+)\.(\w+)\b", sql)
    checked = [(a, c) for a, c in refs if a in aliases]
    bad = sorted({f"{a}.{c}" for a, c in checked if c not in SCHEMA[aliases[a]]})
    return len(checked), bad


{t: sorted(cols) for t, cols in SCHEMA.items()}

{'patients': ['city',
  'date_of_birth',
  'insurance_type',
  'patient_id',
  'patient_name'],
 'appointments': ['appointment_date',
  'appointment_id',
  'appointment_status',
  'appointment_type',
  'clinic_id',
  'patient_id',
  'provider_id'],
 'providers': ['provider_id', 'provider_name', 'specialty'],
 'clinics': ['city', 'clinic_id', 'clinic_name', 'region'],
 'reporting_context': ['context_id',
  'data_end_date',
  'data_start_date',
  'note',
  'primary_date_column',
  'recent_30_day_start',
  'recent_3_month_start',
  'recent_60_day_start',
  'recent_6_month_start',
  'reporting_date']}

## Task 1: Identify (Revisit and Clarify the Problem)

**Original business question (previous lab):** Which patients have missed two or more appointments (`no_show` or `canceled`) in the most recent 3-month reporting window, and how large a share of their visits did they miss?

**How key terms were defined:**
- **Missed appointment:** `appointment_status IN ('no_show', 'canceled')`
- **Recent window:** `appointment_date` from `reporting_context.recent_3_month_start` (2025-03-29) to `reporting_context.reporting_date` (2025-06-29), never `DATE('now')`
- **High-risk patient:** 2 or more missed appointments in the window

**What the output was supposed to produce:** a list the clinic operations team could use to **prioritize reminder calls and rescheduling outreach**. That purpose is the real test: the list has to point staff at patients who need contact *now*.

**Updated business problem statement (refined, same question):**
> Which patients have **2 or more missed appointments** (`no_show` or `canceled`) in the reporting period, and of those, which have **not returned for a completed appointment since their most recent miss**, so outreach staff can contact the patients who are still disengaged first?

The refinement keeps the same healthcare question and definition of a missed appointment. It adds what the original lacked to support a decision: whether the patient has already come back.

**Definition of expected output:** one row per high-risk patient with identifiers, separate `no_show` / `canceled` / total missed counts, missed rate, the date of the last miss, the date / status / type of the last appointment, a 0/1 `returned_after_last_miss` flag and an `outreach_status` label (`Needs outreach` / `Monitor`). Patients who need outreach are listed first.

**What "correct and reliable" means here.** A reliable result must:
1. **Be schema-valid:** only real tables and columns, joined on the documented keys, with no rows lost or duplicated.
2. **Be logically correct:** exact status values, a date window anchored to `reporting_context`, totals that include every appointment, no NULLs in fields staff will act on.
3. **Match the business definitions:** high-risk = 2+ missed; no-shows and cancellations visible separately.
4. **Be complete:** the window must not silently drop the patients the team cares about most.
5. **Be reproducible:** an independent method (pandas) gives the same answer, and each flag can be checked against a patient's actual appointment history.

A query can run, match its prompt and still fail point 4 or 5, so passing "it ran" is not enough.

## Task 2: Assemble (Define Validation Criteria)

This checklist is applied to the original SQL in Task 3 and to the revised SQL in Task 7.

| ID | Category | Check | How it is tested |
|---|---|---|---|
| S1 | Schema accuracy | Every table and `alias.column` exists in the schema (no hallucinations) | `invalid_column_refs()` against `PRAGMA table_info` |
| S2 | Schema accuracy | Joins use documented keys and do not drop or duplicate appointment rows | Row counts before/after join |
| S3 | Schema accuracy | `reporting_context` is used only for date boundaries, never joined | Read SQL |
| L1 | Logical correctness | Date window anchored to `reporting_context` with both bounds; no `DATE('now')` | Read SQL + filter-by-filter counts |
| L2 | Logical correctness | Exact status values (`'no_show'`, `'canceled'`, `'completed'`) | Compare to distinct values in data |
| L3 | Logical correctness | **Grouping:** exactly one row per patient (`GROUP BY` on the patient key); `total_appointments` counts all statuses; `missed = no_show + canceled` | Assert unique `patient_id` + count consistency on output |
| L4 | Logical correctness | Calculations are float-safe and no NULLs appear in decision columns | Assert on output |
| B1 | Business alignment | High-risk = 2+ missed in the reporting period | Read SQL (`HAVING` / `WHERE`) |
| B2 | Business alignment | Output tells staff who needs outreach **now** (has the patient returned?) | Output columns + patient timelines |
| B3 | Business alignment | No-shows and cancellations reported separately | Output columns |
| C1 | Completeness | Window does not hide patients who stopped booking appointments | Sensitivity test: 3-month vs 6-month window |
| C2 | Completeness | Deterministic order and tie-breaking (sorting and "last appointment" lookup) | Read SQL |
| R1 | Risk: assumptions | Every business assumption is written down, not left implicit | Task 4 issue log |
| R2 | Risk: edge cases | Patients with no completed visit, boundary dates, same-day visits, small denominators | Targeted tests |
| R3 | Risk: reproducibility | Independent pandas rebuild matches the SQL exactly | `DataFrame.equals` |

## Task 3: Execute (Review Existing SQL Output)

The existing SQL is the **final revised query from the previous lab**. (That lab's first AI draft used `DATE('now')` and `'no-show'` and returned 0 rows. Those problems were already fixed and are not repeated here.)

```sql
WITH window_appointments AS (
    SELECT
        a.appointment_id,
        a.patient_id,
        a.appointment_date,
        a.appointment_status
    FROM appointments AS a
    WHERE a.appointment_date >= (SELECT recent_3_month_start FROM reporting_context)
      AND a.appointment_date <= (SELECT reporting_date FROM reporting_context)
)
SELECT
    p.patient_id,
    p.patient_name,
    p.insurance_type,
    COUNT(w.appointment_id) AS total_appointments,
    SUM(CASE WHEN w.appointment_status = 'no_show' THEN 1 ELSE 0 END) AS no_show_count,
    SUM(CASE WHEN w.appointment_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_count,
    SUM(CASE WHEN w.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END) AS missed_count,
    ROUND(
        100.0 * SUM(CASE WHEN w.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END)
        / COUNT(w.appointment_id),
        1
    ) AS missed_rate_pct,
    MAX(w.appointment_date) AS last_appointment_date
FROM patients AS p
JOIN window_appointments AS w
    ON p.patient_id = w.patient_id
GROUP BY p.patient_id, p.patient_name, p.insurance_type
HAVING SUM(CASE WHEN w.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END) >= 2
ORDER BY missed_count DESC, missed_rate_pct DESC, p.patient_name ASC;
```

In [3]:
original_sql = """
WITH window_appointments AS (
    SELECT
        a.appointment_id,
        a.patient_id,
        a.appointment_date,
        a.appointment_status
    FROM appointments AS a
    WHERE a.appointment_date >= (SELECT recent_3_month_start FROM reporting_context)
      AND a.appointment_date <= (SELECT reporting_date FROM reporting_context)
)
SELECT
    p.patient_id,
    p.patient_name,
    p.insurance_type,
    COUNT(w.appointment_id) AS total_appointments,
    SUM(CASE WHEN w.appointment_status = 'no_show' THEN 1 ELSE 0 END) AS no_show_count,
    SUM(CASE WHEN w.appointment_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_count,
    SUM(CASE WHEN w.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END) AS missed_count,
    ROUND(
        100.0 * SUM(CASE WHEN w.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END)
        / COUNT(w.appointment_id),
        1
    ) AS missed_rate_pct,
    MAX(w.appointment_date) AS last_appointment_date
FROM patients AS p
JOIN window_appointments AS w
    ON p.patient_id = w.patient_id
GROUP BY p.patient_id, p.patient_name, p.insurance_type
HAVING SUM(CASE WHEN w.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END) >= 2
ORDER BY missed_count DESC, missed_rate_pct DESC, p.patient_name ASC;
"""
original_result = pd.read_sql(original_sql, conn)
original_result

,patient_id,patient_name,insurance_type,total_appointments,no_show_count,canceled_count,missed_count,missed_rate_pct,last_appointment_date
0,28,Brandon Carter,HMO,4,2,1,3,75.0,2025-06-05
1,17,Quinn Lewis,Medicaid,2,2,0,2,100.0,2025-04-25
2,39,Maya Stewart,Medicaid,3,2,0,2,66.7,2025-05-28
3,46,Thomas Bell,Medicare,3,2,0,2,66.7,2025-06-07


The output *looks* right. Before trusting it, here is the full appointment history for each flagged patient:

In [4]:
def timelines(patient_ids):
    ids = ",".join(str(int(i)) for i in patient_ids)
    return pd.read_sql(f"""
        SELECT a.patient_id, p.patient_name, a.appointment_date, a.appointment_status, a.appointment_type
        FROM appointments AS a
        JOIN patients AS p ON p.patient_id = a.patient_id
        WHERE a.patient_id IN ({ids})
        ORDER BY a.patient_id, a.appointment_date;
    """, conn)


timelines(original_result["patient_id"])

,patient_id,patient_name,appointment_date,appointment_status,appointment_type
0,17,Quinn Lewis,2025-01-21,completed,screening
1,17,Quinn Lewis,2025-04-01,no_show,screening
2,17,Quinn Lewis,2025-04-25,no_show,follow_up
3,28,Brandon Carter,2025-04-10,no_show,checkup
4,28,Brandon Carter,2025-05-10,no_show,checkup
5,28,Brandon Carter,2025-05-23,completed,follow_up
6,28,Brandon Carter,2025-06-05,canceled,follow_up
7,39,Maya Stewart,2025-04-02,no_show,consult
8,39,Maya Stewart,2025-04-28,no_show,consult
9,39,Maya Stewart,2025-05-28,completed,follow_up


In [5]:
n_refs, bad = invalid_column_refs(original_sql)
print(f"S1: checked {n_refs} column references, invalid: {bad or 'none'}")

S1: checked 14 column references, invalid: none


**Checklist applied to the original SQL**

| ID | Result | Evidence |
|---|---|---|
| S1 | ✅ Pass | All column references valid; no hallucinated tables or columns |
| S2 | ✅ Pass | Inner join on `patient_id`, which preserves all 58 window appointments (verified in the previous lab) |
| S3 | ✅ Pass | `reporting_context` used only in scalar subqueries |
| L1 | ✅ Pass | Anchored to `recent_3_month_start` / `reporting_date` |
| L2 | ✅ Pass | Exact values `'no_show'`, `'canceled'` |
| L3 | ✅ Pass | One row per patient; conditional aggregation keeps completed visits in the total |
| L4 | ✅ Pass | `100.0 *` avoids integer division; no NULLs |
| B1 | ✅ Pass | `HAVING missed >= 2` |
| B2 | ❌ **Fail** | Timelines show **Maya Stewart and Thomas Bell already came back** and completed a follow-up after their no-shows, yet they get the same priority as patients who never returned. `last_appointment_date` hides this, because it doesn't say *what happened* on that date |
| B3 | ✅ Pass | Separate `no_show_count` and `canceled_count` |
| C1 | ❌ **Fail** | Not tested in the previous lab. See Task 4: the 3-month window hides a fully disengaged patient |
| C2 | ✅ Pass | Three-level sort with name tiebreaker |
| R1 | ⚠️ Partial | Treating `canceled` the same as `no_show` is a business assumption that was never confirmed |
| R2 | ⚠️ Partial | Edge cases not tested; small denominators produce rates such as 100% from 2 visits |
| R3 | ✅ Pass | pandas rebuild matched in the previous lab |

**Summary:** the original SQL passes every *technical* check but fails the *business* checks. It runs, matches its prompt and contains no hallucinations, yet it would point outreach staff at the wrong patients.

### Initial SQL Review Notes

| Review question | Verdict | Notes |
|---|---|---|
| Uses correct tables and joins? | ✅ Yes | `patients` INNER JOIN `appointments` on `patient_id`, the documented key. `reporting_context` is correctly *not* joined; it is only read through scalar subqueries. No appointments are lost or duplicated |
| References valid columns? | ✅ Yes | All 14 column references exist in the schema (`invalid_column_refs` check above). No hallucinated tables or columns |
| Applies correct filters and constraints? | ⚠️ Partly | Status values are spelled exactly and the dates are anchored to `reporting_context`. **But** the 3-month window is applied to `appointment_date`, so a patient who stops booking falls out of it. That constraint is technically correct but wrong for the business purpose |
| Implements the intended logic? | ⚠️ Partly | Counting and grouping are correct (conditional `SUM(CASE ...)`, totals include completed visits, `HAVING >= 2`). **But** the intended purpose is outreach, and nothing in the logic checks whether a patient came back after missing |
| Matches the expected output? | ⚠️ Partly | It returns exactly the columns the previous prompt asked for, and it matches the validation notebook's expected-output check. **But** those 4 rows are not the list outreach staff actually need (see below) |

**What works correctly**
- Schema use is clean: right tables, right join key, valid columns, no invented fields.
- The date window is anchored to `reporting_context`, not `DATE('now')`, and includes both a start and an end bound.
- The exact stored values `'no_show'` and `'canceled'` are used.
- Aggregation is correct: completed visits stay in `total_appointments`, so `missed_rate_pct` is a true rate and `100.0 *` prevents integer division.
- No-shows and cancellations are counted separately, and the sort is deterministic, with a name tiebreaker.

**What is unclear or incorrect**
- **Incorrect for the business goal: no re-engagement check.** Maya Stewart and Thomas Bell each completed a follow-up *after* their no-shows (see timelines), yet they are ranked the same as patients who never returned. Half of the list would waste outreach calls.
- **Subtle completeness gap.** A window keyed on appointment dates can't see patients who have stopped booking. Task 4 shows a patient this hides.
- **Unclear output field.** `last_appointment_date` doesn't say whether that visit was completed or missed. For Brandon Carter it is a cancellation; for Thomas Bell it is a completed visit. They look the same in the output.
- **Unconfirmed assumption.** `canceled` counts as a full miss, but nobody confirmed that cancellations (which may come with notice) should weigh the same as no-shows.
- **Misleading metric.** `missed_rate_pct` = 100% for Quinn Lewis comes from just 2 appointments, which looks more certain than it is.
- **Hint confirmed:** the query runs, matches its prompt and passes the validation notebook's check, and it is still wrong for the decision it was built to support.

## Task 4: Execute (Identify Issues and Risks)

In [6]:
# C1 sensitivity test: who has 2+ missed appointments in the 3-month window vs the 6-month window?
window_test = pd.read_sql("""
SELECT
    p.patient_id,
    p.patient_name,
    SUM(CASE WHEN a.appointment_status IN ('no_show', 'canceled')
              AND a.appointment_date >= (SELECT recent_3_month_start FROM reporting_context) THEN 1 ELSE 0 END) AS missed_3_month,
    SUM(CASE WHEN a.appointment_status IN ('no_show', 'canceled')
              AND a.appointment_date >= (SELECT recent_6_month_start FROM reporting_context) THEN 1 ELSE 0 END) AS missed_6_month,
    MAX(CASE WHEN a.appointment_status = 'completed' THEN a.appointment_date END) AS last_completed_date,
    MAX(a.appointment_date) AS last_appointment_date
FROM patients AS p
JOIN appointments AS a ON p.patient_id = a.patient_id
WHERE a.appointment_date <= (SELECT reporting_date FROM reporting_context)
GROUP BY p.patient_id, p.patient_name
HAVING missed_6_month >= 2
ORDER BY missed_3_month, p.patient_id;
""", conn)
window_test

,patient_id,patient_name,missed_3_month,missed_6_month,last_completed_date,last_appointment_date
0,3,Carlos Rivera,0,2,2025-06-01,2025-06-01
1,8,Hannah Davis,0,2,2025-06-03,2025-06-03
2,20,Tyler Allen,1,2,None,2025-03-29
3,17,Quinn Lewis,2,2,2025-01-21,2025-04-25
4,39,Maya Stewart,2,2,2025-05-28,2025-05-28
5,46,Thomas Bell,2,2,2025-06-07,2025-06-07
6,28,Brandon Carter,3,3,2025-05-23,2025-06-05


In [7]:
timelines([20])

,patient_id,patient_name,appointment_date,appointment_status,appointment_type
0,20,Tyler Allen,2025-01-29,no_show,checkup
1,20,Tyler Allen,2025-03-29,canceled,checkup


**Issue log**

| # | Issue | Type | Why it matters / impact on decisions | Severity |
|---|---|---|---|---|
| 1 | **No re-engagement signal.** The query counts misses but never checks whether the patient came back afterward | Weak business logic | 2 of the 4 flagged patients (Maya Stewart, Thomas Bell) **already completed a follow-up visit** after their no-shows. Staff would spend half their outreach calls on patients who have returned, while Quinn Lewis (2 no-shows, no completed visit since) is ranked only 2nd | **High** |
| 2 | **The 3-month window hides patients who stopped booking.** The window is applied to `appointment_date`, so a patient who stops scheduling drops out of it | Missing condition / survivorship bias | **Tyler Allen** no-showed on 2025-01-29, canceled on 2025-03-29 and has **no appointment since**. Only 1 miss falls in the 3-month window, so Tyler is excluded. The most disengaged patient is the one the query cannot see | **High** |
| 3 | **`canceled` is treated as equal to `no_show`** | Unconfirmed assumption | A cancellation with notice lets the clinic refill the slot; a no-show does not. The schema has no cancellation-reason or notice field, so the data can't separate them. Ranking only on the combined count can overstate risk | Medium |
| 4 | **Rates on small denominators.** `missed_rate_pct` can read 100% from just 2 appointments | Misleading metric | A rate looks more precise than it is; if anyone sorts by it, patients with 2 visits outrank patients with more evidence | Medium |
| 5 | **Ambiguous `last_appointment_date`.** It gives the latest date without saying whether that visit was completed or missed | Misleading output | Staff can't tell from the row whether the patient's last contact went well | Low |
| 6 | **Edge cases never tested.** No test for a patient with zero completed visits, a miss exactly on the window boundary (Tyler's 2025-03-29 cancel equals `recent_3_month_start`), or same-day visits | Validation gap | Bugs in exactly these cases would go unnoticed (Task 5 shows one appearing) | Medium |

**Hallucination check:** no invented tables or columns (S1 passed). The risks in this workflow are **assumptions and missing conditions**, not hallucinations. They are harder to catch because the SQL looks correct.

## Task 5: Execute (Build a Prompt Chain)

Each prompt has one purpose, and the output of each step feeds the next.

| Step | Purpose | Input | Output |
|---|---|---|---|
| Prompt 1 | **Clarify**: restate the task and surface assumptions before any SQL | Business goal, original SQL, issues 1–6 | Restated task, assumption list, questions |
| Prompt 2 | **Generate**: write SQL from the confirmed definitions | Answers to Prompt 1's questions + schema | SQL draft (v2) |
| Prompt 3 | **Review**: audit the draft against the checklist and its real output | v2 SQL, its output, checklist | List of defects |
| Prompt 4 | **Refine**: fix only the defects found in Prompt 3 | Review findings | Final SQL (v3) |

### Prompt 1: Clarify the task

```text
I am a healthcare data analyst validating a SQLite query before my team uses it for outreach
decisions. Do NOT write SQL yet.

Goal: give clinic outreach staff a prioritized list of patients with repeated missed
appointments so they can call the patients who are still disengaged first.

Here is the current query: [original SQL from Task 3]

Validation found these problems:
1. It does not check whether a patient completed an appointment after their misses.
2. The 3-month window drops patients who stopped booking (one patient with 2 misses and no
   visit since is excluded).
3. canceled is weighted the same as no_show.
4. last_appointment_date does not say what happened at that visit.

Restate the task in your own words, list every assumption the new query will need, and ask
me any questions whose answers would change the SQL.
```

**AI response (saved as returned):**

> **Restated task:** Build a patient-level outreach list. Include patients with 2+ missed appointments (`no_show` or `canceled`) in the reporting period. For each, show whether they returned for a completed visit after their most recent miss, and sort so patients who have *not* returned come first.
>
> **Assumptions the query will need:**
> 1. "Returned" means a `completed` appointment dated after the patient's latest `no_show`/`canceled` date.
> 2. The reporting period is anchored to `reporting_context`, not the current date.
> 3. `canceled` still counts as missed, but `no_show` should be shown separately and weighted higher when ranking.
> 4. "Last appointment" means the latest `appointment_date` in the period for that patient.
>
> **Questions:**
> 1. Which window should be used? `recent_6_month_start` would capture patients whose misses happened earlier and who have not booked since.
> 2. Should patients who have returned be removed or kept with a lower priority?
> 3. Do you want a text label for the outreach decision, or only a 0/1 flag?
> 4. If two appointments fall on the same date, which counts as the last one?

### Prompt 2: Generate SQL

```text
Answers: (1) Use recent_6_month_start through reporting_date from reporting_context.
(2) Keep returned patients but sort them after patients who have not returned.
(3) A 0/1 flag is fine for now. (4) Use the latest date.

Write one SQLite query using ONLY these tables and columns:
patients(patient_id, patient_name, insurance_type)
appointments(appointment_id, patient_id, appointment_date 'YYYY-MM-DD', appointment_status)
  -- appointment_status values: 'completed', 'canceled', 'no_show'
reporting_context(recent_6_month_start, reporting_date) -- 1 row, scalar subqueries only
Join patients.patient_id = appointments.patient_id. Do not use DATE('now').

Return one row per patient with 2+ missed appointments (no_show or canceled) in the window:
patient_id, patient_name, insurance_type, total_appointments, no_show_count, canceled_count,
missed_count, missed_rate_pct (1 decimal), last_missed_date, last_appointment_status,
returned_after_last_miss (1 if a completed appointment is dated after the last missed
appointment, otherwise 0).
Sort: patients who have not returned first, then missed_count descending.
```

**AI-generated SQL (v2, saved as returned):**

```sql
SELECT
    p.patient_id,
    p.patient_name,
    p.insurance_type,
    COUNT(a.appointment_id) AS total_appointments,
    SUM(CASE WHEN a.appointment_status = 'no_show' THEN 1 ELSE 0 END) AS no_show_count,
    SUM(CASE WHEN a.appointment_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_count,
    SUM(CASE WHEN a.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END) AS missed_count,
    ROUND(100.0 * SUM(CASE WHEN a.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END)
          / COUNT(a.appointment_id), 1) AS missed_rate_pct,
    MAX(CASE WHEN a.appointment_status IN ('no_show', 'canceled') THEN a.appointment_date END) AS last_missed_date,
    (SELECT a2.appointment_status
     FROM appointments AS a2
     WHERE a2.patient_id = p.patient_id
       AND a2.appointment_date BETWEEN (SELECT recent_6_month_start FROM reporting_context)
                                   AND (SELECT reporting_date FROM reporting_context)
     ORDER BY a2.appointment_date DESC
     LIMIT 1) AS last_appointment_status,
    MAX(CASE WHEN a.appointment_status = 'completed' THEN a.appointment_date END)
        > MAX(CASE WHEN a.appointment_status IN ('no_show', 'canceled') THEN a.appointment_date END)
        AS returned_after_last_miss
FROM patients AS p
JOIN appointments AS a
    ON p.patient_id = a.patient_id
WHERE a.appointment_date >= (SELECT recent_6_month_start FROM reporting_context)
  AND a.appointment_date <= (SELECT reporting_date FROM reporting_context)
GROUP BY p.patient_id, p.patient_name, p.insurance_type
HAVING SUM(CASE WHEN a.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END) >= 2
ORDER BY returned_after_last_miss ASC, missed_count DESC;
```

In [8]:
v2_sql = """
SELECT
    p.patient_id,
    p.patient_name,
    p.insurance_type,
    COUNT(a.appointment_id) AS total_appointments,
    SUM(CASE WHEN a.appointment_status = 'no_show' THEN 1 ELSE 0 END) AS no_show_count,
    SUM(CASE WHEN a.appointment_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_count,
    SUM(CASE WHEN a.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END) AS missed_count,
    ROUND(100.0 * SUM(CASE WHEN a.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END)
          / COUNT(a.appointment_id), 1) AS missed_rate_pct,
    MAX(CASE WHEN a.appointment_status IN ('no_show', 'canceled') THEN a.appointment_date END) AS last_missed_date,
    (SELECT a2.appointment_status
     FROM appointments AS a2
     WHERE a2.patient_id = p.patient_id
       AND a2.appointment_date BETWEEN (SELECT recent_6_month_start FROM reporting_context)
                                   AND (SELECT reporting_date FROM reporting_context)
     ORDER BY a2.appointment_date DESC
     LIMIT 1) AS last_appointment_status,
    MAX(CASE WHEN a.appointment_status = 'completed' THEN a.appointment_date END)
        > MAX(CASE WHEN a.appointment_status IN ('no_show', 'canceled') THEN a.appointment_date END)
        AS returned_after_last_miss
FROM patients AS p
JOIN appointments AS a
    ON p.patient_id = a.patient_id
WHERE a.appointment_date >= (SELECT recent_6_month_start FROM reporting_context)
  AND a.appointment_date <= (SELECT reporting_date FROM reporting_context)
GROUP BY p.patient_id, p.patient_name, p.insurance_type
HAVING SUM(CASE WHEN a.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END) >= 2
ORDER BY returned_after_last_miss ASC, missed_count DESC;
"""
v2_result = pd.read_sql(v2_sql, conn)
v2_result

,patient_id,patient_name,insurance_type,total_appointments,no_show_count,canceled_count,missed_count,missed_rate_pct,last_missed_date,last_appointment_status,returned_after_last_miss
0,20,Tyler Allen,HMO,2,1,1,2,100.0,2025-03-29,canceled,NaN
1,28,Brandon Carter,HMO,4,2,1,3,75.0,2025-06-05,canceled,0.0
2,17,Quinn Lewis,Medicaid,3,2,0,2,66.7,2025-04-25,no_show,0.0
3,3,Carlos Rivera,Medicare,4,2,0,2,50.0,2025-02-14,completed,1.0
4,8,Hannah Davis,HMO,3,2,0,2,66.7,2025-03-06,completed,1.0
5,39,Maya Stewart,Medicaid,3,2,0,2,66.7,2025-04-28,completed,1.0
6,46,Thomas Bell,Medicare,3,2,0,2,66.7,2025-05-17,completed,1.0


### Prompt 3: Review the SQL

```text
Act as a SQL reviewer, not a generator. Review the query below against this checklist:
schema validity, join correctness, date window, exact status values, aggregation, NULL
handling, deterministic ordering, and whether the output supports the outreach decision.
Here is the query and the output it returned when I ran it: [v2 SQL + v2 output table].
List every defect with the line it comes from and its impact. Do not rewrite the query yet.
```

**AI response (saved as returned):**

> **Review findings:**
> 1. **NULL flag (defect).** `returned_after_last_miss` is a bare comparison, `MAX(completed date) > MAX(missed date)`. If a patient has **no completed appointment**, the left side is NULL and the result is NULL, not 0. Tyler Allen shows `NaN`. The prompt required 0/1, and any later filter such as `WHERE returned_after_last_miss = 0` would silently drop the most disengaged patient.
> 2. **Sort works by accident.** Tyler appears first only because SQLite sorts NULLs first in ascending order. Other databases (or a `DESC` sort) would move Tyler to the bottom.
> 3. **No tiebreak for "last appointment".** The correlated subquery uses `ORDER BY appointment_date DESC LIMIT 1`. If a patient had two appointments on the same day, the result would be arbitrary. Add `appointment_id DESC`.
> 4. **Duplicated window logic.** The date window is written twice (main `WHERE` and the subquery). If someone edits one and not the other, the counts and the last status will disagree. Define the window once in a CTE.
> 5. **Incomplete for the decision.** `last_appointment_date` and `last_appointment_type` are missing (staff need to know whether the missed visit was a follow-up), and there is no readable action label.
> 6. **Ranking.** After the flag, the sort uses `missed_count` only, so `no_show` is not weighted higher as agreed in Prompt 1. Add `no_show_count DESC` and a name tiebreaker.
>
> **No issues found with:** table and column names (all valid), the join key, the `reporting_context` anchoring and the status spellings.

### Prompt 4: Refine the SQL

```text
Revise the query to fix every finding from your review, and change nothing else:
1. Make returned_after_last_miss strictly 0 or 1. A patient with no completed appointment = 0.
2. Do not rely on NULL sort order.
3. Break same-day ties for the last appointment with appointment_id DESC.
4. Define the date window once in a CTE and reuse it.
5. Add last_appointment_date, last_appointment_type and an outreach_status column:
   'Needs outreach' when returned_after_last_miss = 0, otherwise 'Monitor'.
6. Sort: returned_after_last_miss ASC, no_show_count DESC, missed_count DESC,
   patient_name ASC.
SQLite only. Same tables, columns, definitions and window as before.
```

## Task 6: Execute (Generate Revised SQL)

**Revised AI-generated SQL (v3, final output of the prompt chain, saved without manual edits):**

```sql
WITH window_appointments AS (
    SELECT
        a.appointment_id,
        a.patient_id,
        a.appointment_date,
        a.appointment_status,
        a.appointment_type
    FROM appointments AS a
    WHERE a.appointment_date >= (SELECT recent_6_month_start FROM reporting_context)
      AND a.appointment_date <= (SELECT reporting_date FROM reporting_context)
),
patient_summary AS (
    SELECT
        w.patient_id,
        COUNT(w.appointment_id) AS total_appointments,
        SUM(CASE WHEN w.appointment_status = 'no_show' THEN 1 ELSE 0 END) AS no_show_count,
        SUM(CASE WHEN w.appointment_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_count,
        SUM(CASE WHEN w.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END) AS missed_count,
        MAX(CASE WHEN w.appointment_status IN ('no_show', 'canceled') THEN w.appointment_date END) AS last_missed_date,
        MAX(CASE WHEN w.appointment_status = 'completed' THEN w.appointment_date END) AS last_completed_date
    FROM window_appointments AS w
    GROUP BY w.patient_id
),
last_appointment AS (
    SELECT
        w.patient_id,
        w.appointment_date AS last_appointment_date,
        w.appointment_status AS last_appointment_status,
        w.appointment_type AS last_appointment_type,
        ROW_NUMBER() OVER (
            PARTITION BY w.patient_id
            ORDER BY w.appointment_date DESC, w.appointment_id DESC
        ) AS rn
    FROM window_appointments AS w
)
SELECT
    p.patient_id,
    p.patient_name,
    p.insurance_type,
    s.total_appointments,
    s.no_show_count,
    s.canceled_count,
    s.missed_count,
    ROUND(100.0 * s.missed_count / s.total_appointments, 1) AS missed_rate_pct,
    s.last_missed_date,
    l.last_appointment_date,
    l.last_appointment_status,
    l.last_appointment_type,
    CASE
        WHEN s.last_completed_date IS NOT NULL
             AND s.last_completed_date > s.last_missed_date THEN 1
        ELSE 0
    END AS returned_after_last_miss,
    CASE
        WHEN s.last_completed_date IS NOT NULL
             AND s.last_completed_date > s.last_missed_date THEN 'Monitor'
        ELSE 'Needs outreach'
    END AS outreach_status
FROM patient_summary AS s
JOIN patients AS p
    ON p.patient_id = s.patient_id
JOIN last_appointment AS l
    ON l.patient_id = s.patient_id
   AND l.rn = 1
WHERE s.missed_count >= 2
ORDER BY
    returned_after_last_miss ASC,
    s.no_show_count DESC,
    s.missed_count DESC,
    p.patient_name ASC;
```

In [9]:
revised_sql = """
WITH window_appointments AS (
    SELECT
        a.appointment_id,
        a.patient_id,
        a.appointment_date,
        a.appointment_status,
        a.appointment_type
    FROM appointments AS a
    WHERE a.appointment_date >= (SELECT recent_6_month_start FROM reporting_context)
      AND a.appointment_date <= (SELECT reporting_date FROM reporting_context)
),
patient_summary AS (
    SELECT
        w.patient_id,
        COUNT(w.appointment_id) AS total_appointments,
        SUM(CASE WHEN w.appointment_status = 'no_show' THEN 1 ELSE 0 END) AS no_show_count,
        SUM(CASE WHEN w.appointment_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_count,
        SUM(CASE WHEN w.appointment_status IN ('no_show', 'canceled') THEN 1 ELSE 0 END) AS missed_count,
        MAX(CASE WHEN w.appointment_status IN ('no_show', 'canceled') THEN w.appointment_date END) AS last_missed_date,
        MAX(CASE WHEN w.appointment_status = 'completed' THEN w.appointment_date END) AS last_completed_date
    FROM window_appointments AS w
    GROUP BY w.patient_id
),
last_appointment AS (
    SELECT
        w.patient_id,
        w.appointment_date AS last_appointment_date,
        w.appointment_status AS last_appointment_status,
        w.appointment_type AS last_appointment_type,
        ROW_NUMBER() OVER (
            PARTITION BY w.patient_id
            ORDER BY w.appointment_date DESC, w.appointment_id DESC
        ) AS rn
    FROM window_appointments AS w
)
SELECT
    p.patient_id,
    p.patient_name,
    p.insurance_type,
    s.total_appointments,
    s.no_show_count,
    s.canceled_count,
    s.missed_count,
    ROUND(100.0 * s.missed_count / s.total_appointments, 1) AS missed_rate_pct,
    s.last_missed_date,
    l.last_appointment_date,
    l.last_appointment_status,
    l.last_appointment_type,
    CASE
        WHEN s.last_completed_date IS NOT NULL
             AND s.last_completed_date > s.last_missed_date THEN 1
        ELSE 0
    END AS returned_after_last_miss,
    CASE
        WHEN s.last_completed_date IS NOT NULL
             AND s.last_completed_date > s.last_missed_date THEN 'Monitor'
        ELSE 'Needs outreach'
    END AS outreach_status
FROM patient_summary AS s
JOIN patients AS p
    ON p.patient_id = s.patient_id
JOIN last_appointment AS l
    ON l.patient_id = s.patient_id
   AND l.rn = 1
WHERE s.missed_count >= 2
ORDER BY
    returned_after_last_miss ASC,
    s.no_show_count DESC,
    s.missed_count DESC,
    p.patient_name ASC;
"""
revised_result = pd.read_sql(revised_sql, conn)
revised_result

,patient_id,patient_name,insurance_type,total_appointments,no_show_count,canceled_count,missed_count,missed_rate_pct,last_missed_date,last_appointment_date,last_appointment_status,last_appointment_type,returned_after_last_miss,outreach_status
0,28,Brandon Carter,HMO,4,2,1,3,75.0,2025-06-05,2025-06-05,canceled,follow_up,0,Needs outreach
1,17,Quinn Lewis,Medicaid,3,2,0,2,66.7,2025-04-25,2025-04-25,no_show,follow_up,0,Needs outreach
2,20,Tyler Allen,HMO,2,1,1,2,100.0,2025-03-29,2025-03-29,canceled,checkup,0,Needs outreach
3,3,Carlos Rivera,Medicare,4,2,0,2,50.0,2025-02-14,2025-06-01,completed,follow_up,1,Monitor
4,8,Hannah Davis,HMO,3,2,0,2,66.7,2025-03-06,2025-06-03,completed,checkup,1,Monitor
5,39,Maya Stewart,Medicaid,3,2,0,2,66.7,2025-04-28,2025-05-28,completed,follow_up,1,Monitor
6,46,Thomas Bell,Medicare,3,2,0,2,66.7,2025-05-17,2025-06-07,completed,follow_up,1,Monitor


## Task 7: Verify (Validate the Revised Workflow)

Each checklist item from Task 2 is reapplied below with code wherever possible. Nothing is assumed correct just because it is the second version.

In [10]:
results = {}

# S1: no hallucinated columns
for name, sql in [("original", original_sql), ("v2", v2_sql), ("revised", revised_sql)]:
    n_refs, bad = invalid_column_refs(sql)
    print(f"S1 {name:8s}: {n_refs} column refs checked, invalid: {bad or 'none'}")
results["S1"] = invalid_column_refs(revised_sql)[1] == []

# S2: join preserves appointment rows in the window
window_rows = pd.read_sql("""
SELECT COUNT(*) AS n FROM appointments
WHERE appointment_date BETWEEN (SELECT recent_6_month_start FROM reporting_context)
                           AND (SELECT reporting_date FROM reporting_context);""", conn).iloc[0, 0]
joined_rows = pd.read_sql("""
SELECT COUNT(*) AS n FROM appointments a JOIN patients p ON a.patient_id = p.patient_id
WHERE a.appointment_date BETWEEN (SELECT recent_6_month_start FROM reporting_context)
                             AND (SELECT reporting_date FROM reporting_context);""", conn).iloc[0, 0]
results["S2"] = window_rows == joined_rows
print(f"S2: window appointments {window_rows}, after join {joined_rows}")

# S3 / L1 / L2: anchored window, exact values, no live dates
results["S3"] = "JOIN reporting_context" not in revised_sql
results["L1"] = "now" not in revised_sql.lower() and "recent_6_month_start" in revised_sql and "reporting_date" in revised_sql
stored = set(pd.read_sql("SELECT DISTINCT appointment_status FROM appointments;", conn)["appointment_status"])
used = set(re.findall(r"'(\w+)'", revised_sql)) & {"no_show", "no-show", "canceled", "completed", "cancelled"}
results["L2"] = used <= stored
print(f"L2: status values used {sorted(used)}, stored {sorted(stored)}")

# L3 / L4: internal consistency, no NULLs
r = revised_result
results["L3"] = bool(r["patient_id"].is_unique
                     and ((r["no_show_count"] + r["canceled_count"]) == r["missed_count"]).all()
                     and (r["missed_count"] <= r["total_appointments"]).all())
results["L4"] = bool(r.notna().all().all() and set(r["returned_after_last_miss"]) <= {0, 1})
print(f"L4: NULLs in revised output: {int(r.isna().sum().sum())} (v2 had {int(v2_result.isna().sum().sum())})")

# B1 / B3
results["B1"] = bool((r["missed_count"] >= 2).all())
results["B3"] = {"no_show_count", "canceled_count"} <= set(r.columns)

S1 original: 14 column refs checked, invalid: none
S1 v2      : 28 column refs checked, invalid: none
S1 revised : 12 column refs checked, invalid: none
S2: window appointments 120, after join 120
L2: status values used ['canceled', 'completed', 'no_show'], stored ['canceled', 'completed', 'no_show']
L4: NULLs in revised output: 0 (v2 had 1)


In [11]:
# R3 + B2: rebuild the whole result independently in pandas and compare exactly
ctx = pd.read_sql("SELECT recent_6_month_start, reporting_date FROM reporting_context;", conn).iloc[0]
appts = pd.read_sql("SELECT * FROM appointments;", conn)
patients = pd.read_sql("SELECT patient_id, patient_name, insurance_type FROM patients;", conn)

w = appts[appts["appointment_date"].between(ctx["recent_6_month_start"], ctx["reporting_date"])].copy()
w["missed"] = w["appointment_status"].isin(["no_show", "canceled"])
last = w.sort_values(["appointment_date", "appointment_id"]).groupby("patient_id").tail(1).set_index("patient_id")

rows = []
for pid, g in w.groupby("patient_id"):
    missed = g[g["missed"]]
    if len(missed) < 2:
        continue
    last_missed = missed["appointment_date"].max()
    completed = g.loc[g["appointment_status"] == "completed", "appointment_date"]
    returned = int(len(completed) > 0 and completed.max() > last_missed)
    rows.append({
        "patient_id": pid,
        "total_appointments": len(g),
        "no_show_count": int((g["appointment_status"] == "no_show").sum()),
        "canceled_count": int((g["appointment_status"] == "canceled").sum()),
        "missed_count": len(missed),
        "missed_rate_pct": round(100.0 * len(missed) / len(g), 1),
        "last_missed_date": last_missed,
        "last_appointment_date": last.loc[pid, "appointment_date"],
        "last_appointment_status": last.loc[pid, "appointment_status"],
        "last_appointment_type": last.loc[pid, "appointment_type"],
        "returned_after_last_miss": returned,
        "outreach_status": "Monitor" if returned else "Needs outreach",
    })

expected = (pd.DataFrame(rows).merge(patients, on="patient_id")
            .sort_values(["returned_after_last_miss", "no_show_count", "missed_count", "patient_name"],
                         ascending=[True, False, False, True])
            [list(revised_result.columns)].reset_index(drop=True))

results["R3"] = expected.equals(revised_result)
print("R3: pandas rebuild matches revised SQL exactly:", results["R3"])

R3: pandas rebuild matches revised SQL exactly: True


In [12]:
# B2: check every flag by hand against the patient's real appointment history
timelines(revised_result["patient_id"])

,patient_id,patient_name,appointment_date,appointment_status,appointment_type
0,3,Carlos Rivera,2025-01-14,no_show,screening
1,3,Carlos Rivera,2025-02-14,no_show,screening
2,3,Carlos Rivera,2025-04-02,completed,follow_up
3,3,Carlos Rivera,2025-06-01,completed,follow_up
4,8,Hannah Davis,2025-02-04,no_show,checkup
5,8,Hannah Davis,2025-03-06,no_show,checkup
6,8,Hannah Davis,2025-06-03,completed,checkup
7,17,Quinn Lewis,2025-01-21,completed,screening
8,17,Quinn Lewis,2025-04-01,no_show,screening
9,17,Quinn Lewis,2025-04-25,no_show,follow_up


In [13]:
# C1 / R2: targeted edge-case tests
tyler = revised_result.set_index("patient_name").loc["Tyler Allen"]
ctx3 = pd.read_sql("SELECT recent_3_month_start FROM reporting_context;", conn).iloc[0, 0]
same_day = pd.read_sql("""SELECT COUNT(*) AS n FROM (SELECT patient_id, appointment_date FROM appointments
                           GROUP BY 1, 2 HAVING COUNT(*) > 1);""", conn).iloc[0, 0]

edge_cases = pd.DataFrame([
    {"edge case": "Patient with NO completed appointment (Tyler Allen)",
     "original": "Excluded from list", "v2": "returned flag = NULL",
     "revised": f"returned = {tyler['returned_after_last_miss']}, {tyler['outreach_status']}"},
    {"edge case": f"Miss exactly on recent_3_month_start ({ctx3})",
     "original": "Counted (inclusive >=) but only 1 miss in window", "v2": "Counted",
     "revised": "Counted; 6-month window includes the earlier no-show too"},
    {"edge case": "Completed visit BETWEEN misses (Brandon Carter: no_show, no_show, completed, canceled)",
     "original": "Ranked 1st, no context", "v2": "returned = 0",
     "revised": "returned = 0 (latest event is a cancel), Needs outreach"},
    {"edge case": f"Same-day appointments for a patient (found in data: {same_day})",
     "original": "n/a", "v2": "LIMIT 1 without tiebreak",
     "revised": "appointment_id DESC tiebreak"},
])
results["C1"] = 20 in set(revised_result["patient_id"])
results["C2"] = "appointment_id DESC" in revised_sql and "patient_name ASC" in revised_sql
results["R2"] = tyler["returned_after_last_miss"] == 0
edge_cases

,edge case,original,v2,revised
0,Patient with NO completed appointment (Tyler A...,Excluded from list,returned flag = NULL,"returned = 0, Needs outreach"
1,Miss exactly on recent_3_month_start (2025-03-29),Counted (inclusive >=) but only 1 miss in window,Counted,Counted; 6-month window includes the earlier n...
2,Completed visit BETWEEN misses (Brandon Carter...,"Ranked 1st, no context",returned = 0,"returned = 0 (latest event is a cancel), Needs..."
3,Same-day appointments for a patient (found in ...,n/a,LIMIT 1 without tiebreak,appointment_id DESC tiebreak


In [14]:
# Overall checklist result for the revised SQL
results["B2"] = {"returned_after_last_miss", "outreach_status", "last_appointment_status"} <= set(revised_result.columns)
results["R1"] = True  # all assumptions documented in Task 4 / Prompt 1
order = ["S1", "S2", "S3", "L1", "L2", "L3", "L4", "B1", "B2", "B3", "C1", "C2", "R1", "R2", "R3"]
assert all(results[k] for k in order), {k: results[k] for k in order if not results[k]}
print("All 15 checklist items pass for the revised SQL.")

All 15 checklist items pass for the revised SQL.


In [15]:
# Before vs after: who would outreach staff call?
before = set(original_result["patient_name"])
after_outreach = set(revised_result.loc[revised_result["outreach_status"] == "Needs outreach", "patient_name"])
pd.DataFrame({
    "patient": sorted(before | set(revised_result["patient_name"])),
}).assign(
    on_original_list=lambda d: d["patient"].isin(before),
    revised_status=lambda d: d["patient"].map(revised_result.set_index("patient_name")["outreach_status"]),
)

,patient,on_original_list,revised_status
0,Brandon Carter,True,Needs outreach
1,Carlos Rivera,False,Monitor
2,Hannah Davis,False,Monitor
3,Maya Stewart,True,Monitor
4,Quinn Lewis,True,Needs outreach
5,Thomas Bell,True,Monitor
6,Tyler Allen,False,Needs outreach


**Validation report**

**Before vs after comparison**

| Checklist area | Original SQL | Revised SQL (v3) |
|---|---|---|
| Schema accuracy (S1–S3) | ✅ Pass | ✅ Pass. All references valid; join keeps all 120 window rows |
| Logical correctness (L1–L4) | ✅ Pass | ✅ Pass. No NULLs (v2 had 1); counts consistent |
| Business alignment (B1–B3) | ❌ B2 fails: no way to tell who has returned | ✅ Pass. `returned_after_last_miss`, last status/type and an `outreach_status` label |
| Completeness (C1–C2) | ❌ C1 fails: 3-month window hides Tyler Allen | ✅ Pass. 6-month anchored window; deterministic ties |
| Risk (R1–R3) | ⚠️ Assumptions implicit, edge cases untested | ✅ Pass. Assumptions logged, edge cases tested, exact pandas match |

**Evidence of improvement**
- **Outreach list is now correct:** *Needs outreach* = **Brandon Carter, Quinn Lewis, Tyler Allen**. Every one of them has a missed appointment as their most recent event (see timelines).
- **2 of the original 4 patients were already re-engaged:** Maya Stewart and Thomas Bell completed follow-up visits after their no-shows. They are now labeled *Monitor* instead of taking up outreach calls.
- **1 hidden patient found:** Tyler Allen (2 misses, never returned) was invisible in the original output.
- **1 defect caught before use:** the review prompt found the NULL flag in v2 that would have dropped Tyler under a `= 0` filter.
- **Independent proof:** a pandas rebuild of all 14 columns matches the SQL output exactly (`DataFrame.equals`), and all 15 checklist items pass in code.

**Remaining limitations**
1. **Cancellation assumption needs business sign-off.** `canceled` still counts as missed, because the schema has no reason or notice field. The no-show count is shown separately and ranked first, but the manager should confirm the definition.
2. **Future bookings are invisible.** The data contains no `scheduled` status, so a *Needs outreach* patient may already have a future appointment booked. Staff should check the scheduling system before calling.
3. **Small denominators.** Patients have only 2–4 appointments, so `missed_rate_pct` is supporting context, not a ranking metric (it is deliberately left out of the sort).
4. **Window equals the full dataset.** `recent_6_month_start` (2024-12-29) is earlier than the first appointment, so this query effectively covers all data. In production, the window should be a parameter that is reviewed when more history accumulates.
5. **Small, fictional sample.** 120 appointments and 50 patients is enough to validate the logic, not to draw clinical conclusions.

**Does the SQL now meet the business goal?** Yes, for its stated purpose: a validated, prioritized outreach list. Limitations 1 and 2 should travel with the results when they are shared.

## Task 8: Verify (Explain My Learning and Improvements)

**The most important issue I found.** I found that the original query was technically correct but answered the wrong operational question. It counted missed appointments without checking whether the patient had already come back, so half the outreach list was patients who had already returned and it also left out the most disengaged patient because a window keyed on appointment dates cannot see people who stop booking appointments.

**How the prompt chain improved the SQL.** It improved the SQL by splitting the work into clarify → generate → review → refine gave each step a single job. The clarify prompt brought assumptions to the surface (what "returned" means, which window to use) before any SQL existed. The review prompt, run against real output, caught a NULL bug in the AI's own draft that a single "write me a query" prompt would have shipped. The refine prompt then fixed only those defects, so every change can be traced back to a specific finding.

**How validation changed my thinking.** In the last lab I counted the query as correct once it ran and matched an expected output. Now I start from the decision the data will drive and ask, "Who would this send staff to call, and is that right?" Looking at individual patient timelines did more than any syntax check. I also learned not to trust the second version just because it was revised: v2 had a new bug, and only testing proved v3.

**How this applies to real-world data workflows.** In healthcare, a query that "looks right" can send outreach to the wrong patients and miss the ones at real risk. The workflow I would use on any AI-generated SQL is: write the validation checklist before generating, separate generating from reviewing, test edge cases on purpose, rebuild key results independently, and share remaining limitations with the results instead of hiding them.

In [16]:
conn.close()
print("Database connection closed.")

Database connection closed.
